# Fine-tune YOLO26s on Unified Dataset (Road + TDUS)

**目标**：从 `tree_yolo26s_halfres/weights/best.pt`（mAP50=0.886）微调，加入 TDUS 城市街道树木域。

**关键配置**：
- `freeze=5`：保留浅层特征，降低路面域遗忘风险
- `lr0=2e-4`：微调学习率
- `rect=False`：允许跨域 mosaic 混合
- `mosaic=0.5`：开启跨域增强

**消融实验**：若 TDUS 准确率未达预期，可将 `freeze=5` 改为 `freeze=0` 重跑，并对比 road-only mAP50。

In [1]:
# Cell 1: Mount Drive + 安装依赖
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics==8.4.56

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Cell 2: 路径配置
import shutil
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/TreeLearn")

# best.pt 来源
DRIVE_BEST_PT  = DRIVE_DIR / "tree_yolo26s_halfres" / "weights" / "best.pt"
# 微调结果保存目录
DRIVE_OUT_DIR  = DRIVE_DIR / "tree_yolo26s_unified_halfres_tdus"
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

# 统一数据集（从 Drive 挂载，或事先解压到本地）
DRIVE_DATA_DIR = DRIVE_DIR / "unified_halfres_tdus"
LOCAL_DATA_DIR = Path("/content/data/unified_halfres_tdus")

print(f"DRIVE_BEST_PT exists: {DRIVE_BEST_PT.exists()}")
print(f"DRIVE_DATA_DIR exists: {DRIVE_DATA_DIR.exists()}")

DRIVE_BEST_PT exists: True
DRIVE_DATA_DIR exists: True


In [4]:
import yaml
yaml_path = LOCAL_DATA_DIR / "data.yaml"

In [3]:
# Cell 3: 复制数据集到本地（Drive I/O 慢，训练时用本地路径）
if not LOCAL_DATA_DIR.exists():
    print("Copying dataset from Drive to local...")
    shutil.copytree(str(DRIVE_DATA_DIR), str(LOCAL_DATA_DIR))
    print(f"Done: {LOCAL_DATA_DIR}")
else:
    print(f"Local dataset already exists: {LOCAL_DATA_DIR}")

# 修正 data.yaml 中的 path 为本地绝对路径
# import yaml
# yaml_path = LOCAL_DATA_DIR / "data.yaml"
data = yaml.safe_load(yaml_path.read_text())
data["path"] = str(LOCAL_DATA_DIR.resolve())
yaml_path.write_text(yaml.dump(data, default_flow_style=False))
print(f"data.yaml path updated to: {data['path']}")

n_train = len(list((LOCAL_DATA_DIR / "images" / "train").glob("*.jpg")))
n_val   = len(list((LOCAL_DATA_DIR / "images" / "val").glob("*.jpg")))
print(f"Train: {n_train}, Val: {n_val}")

Copying dataset from Drive to local...
Done: /content/data/unified_halfres_tdus
data.yaml path updated to: /content/data/unified_halfres_tdus
Train: 5247, Val: 2477


In [5]:
# Cell 4: Resume 检查 + 训练
from ultralytics import YOLO

DRIVE_WEIGHTS_DIR = DRIVE_OUT_DIR / "weights"
DRIVE_LAST_PT     = DRIVE_WEIGHTS_DIR / "last.pt"
LOCAL_LAST_PT     = Path("/content/last.pt")
LOCAL_BEST_PT     = Path("/content/best.pt")

RUN_NAME = "tree_yolo26s_unified_halfres_tdus"

if DRIVE_LAST_PT.exists():
    print(f"Resuming from {DRIVE_LAST_PT}")
    shutil.copy2(DRIVE_LAST_PT, LOCAL_LAST_PT)
    # resume=True 时从 checkpoint 恢复所有训练参数，不重传
    model = YOLO(str(LOCAL_LAST_PT))
    model.train(resume=True)
else:
    print(f"Starting fresh fine-tune from {DRIVE_BEST_PT}")
    shutil.copy2(DRIVE_BEST_PT, LOCAL_BEST_PT)
    model = YOLO(str(LOCAL_BEST_PT))
    model.train(
        data=str(yaml_path),
        epochs=50,
        imgsz=1280,
        batch=24,              # autobatch
        freeze=5,              # 保留浅层特征；可消融为 freeze=0
        lr0=2e-4,
        lrf=0.01,
        cos_lr=True,
        patience=15,
        rect=False,            # 跨域 mosaic 的必要条件
        mosaic=0.5,
        scale=0.5,
        translate=0.15,
        degrees=10.0,
        fliplr=0.5,
        flipud=0.0,
        mixup=0.0,
        close_mosaic=10,
        project="/content/runs/detect",
        name=RUN_NAME,
        device=0,
        workers=4,
        cache=True,
        deterministic=False,
        save=True,
    )

Starting fresh fine-tune from /content/drive/MyDrive/TreeLearn/tree_yolo26s_halfres/weights/best.pt
New https://pypi.org/project/ultralytics/8.4.58 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data/unified_halfres_tdus/data.yaml, degrees=10.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=5, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_r

In [ ]:
# Cell 5: Road-only val mAP50 評估
import re

# best.pt: 优先用 Drive 保存的权重（session 重置后 runs/ 目录消失）
LOCAL_EVAL_PT = Path("/content/eval_best.pt")
if not LOCAL_EVAL_PT.exists():
    shutil.copy2(DRIVE_WEIGHTS_DIR / "best.pt", LOCAL_EVAL_PT)
    print(f"Copied best.pt from Drive → {LOCAL_EVAL_PT}")

# Road val 帧：先查本地统一数据集，不存在则从 Drive 读取（约 5-10 分钟）
LOCAL_ROAD_VAL_DIR = Path("/content/data/road_val_only")
LOCAL_ROAD_VAL_IMG = LOCAL_ROAD_VAL_DIR / "images" / "val"
LOCAL_ROAD_VAL_LBL = LOCAL_ROAD_VAL_DIR / "labels" / "val"
LOCAL_ROAD_VAL_IMG.mkdir(parents=True, exist_ok=True)
LOCAL_ROAD_VAL_LBL.mkdir(parents=True, exist_ok=True)

road_pattern = re.compile(r'^frame_\d+\.jpg$')

if len(list(LOCAL_ROAD_VAL_IMG.glob("*.jpg"))) == 0:
    candidates = [
        (LOCAL_DATA_DIR  / "images" / "val", LOCAL_DATA_DIR  / "labels" / "val"),
        (DRIVE_DATA_DIR  / "images" / "val", DRIVE_DATA_DIR  / "labels" / "val"),
    ]
    for src_img_dir, src_lbl_dir in candidates:
        if src_img_dir.exists():
            print(f"Extracting road frames from {src_img_dir} ...")
            for img_path in src_img_dir.glob("*.jpg"):
                if road_pattern.match(img_path.name):
                    shutil.copy2(img_path, LOCAL_ROAD_VAL_IMG / img_path.name)
                    lbl = src_lbl_dir / (img_path.stem + ".txt")
                    if lbl.exists():
                        shutil.copy2(lbl, LOCAL_ROAD_VAL_LBL / lbl.name)
            break

n_road = len(list(LOCAL_ROAD_VAL_IMG.glob("*.jpg")))
print(f"Road val frames: {n_road}  (expected ~2082)")

road_yaml_path = LOCAL_ROAD_VAL_DIR / "data.yaml"
road_yaml_path.write_text(yaml.dump({
    "path":  str(LOCAL_ROAD_VAL_DIR.resolve()),
    "train": "images/val",
    "val":   "images/val",
    "nc": 1,
    "names": ["tree"],
}, default_flow_style=False))

eval_model = YOLO(str(LOCAL_EVAL_PT))
metrics = eval_model.val(data=str(road_yaml_path), imgsz=1280, batch=16, device=0)
road_map50 = metrics.box.map50
print(f"\nRoad-only mAP50: {road_map50:.4f}  (target: \u226500.85, baseline: 0.886)")

(DRIVE_OUT_DIR / "road_only_map50.txt").write_text(f"road_only_mAP50={road_map50:.6f}\n")
print("Saved road_only_map50.txt")


In [ ]:
# Cell 6: 保存权重 + results.csv 回 Drive
run_dir     = Path(f"/content/runs/detect/{RUN_NAME}")
dst_weights = DRIVE_OUT_DIR / "weights"
dst_weights.mkdir(parents=True, exist_ok=True)

saved_any = False
for fname in ("best.pt", "last.pt"):
    src = run_dir / "weights" / fname
    if src.exists():
        shutil.copy2(src, dst_weights / fname)
        size_mb = src.stat().st_size / 1024 / 1024
        print(f"Saved {fname} → {dst_weights / fname}  ({size_mb:.1f} MB)")
        saved_any = True
    else:
        print(f"WARNING: {src} not found, skipping")

if not saved_any:
    raise RuntimeError(
        f"No weights found under {run_dir / 'weights'}. "
        "Training may not have completed — check the training cell output."
    )

# results.csv 位于 run_dir 根目录（非 weights/ 子目录）
results_src = run_dir / "results.csv"
if results_src.exists():
    shutil.copy2(results_src, DRIVE_OUT_DIR / "results.csv")
    print(f"Saved results.csv → {DRIVE_OUT_DIR / 'results.csv'}")
else:
    print(f"WARNING: results.csv not found at {results_src}")

# 验证 Drive 上实际可见的文件
print("\nDrive 目录内容验证:")
for f in sorted(dst_weights.iterdir()):
    print(f"  {f.name}  {f.stat().st_size / 1024 / 1024:.1f} MB")
print("All artifacts saved to Drive.")